# 03 · Parte 2: RAG y agente con herramientas

Este cuaderno muestra la recuperación sobre el corpus, las herramientas y el loop agéntico. Sin `LLM_API_KEY` se usa el modo de respaldo (router determinista); con clave se ejecuta el LLM real.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e6e3", "font.size": 10})
AZUL, NARANJA, AQUA, AMARILLO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
FIG = ROOT / "reports" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
from aire.rag import Retriever
r = Retriever(); print(r.backend, len(r.chunks), 'fragmentos')
pd.DataFrame(r.search('¿puedo hacer ejercicio si el aire está naranja?', k=3))[['cita','seccion','puntaje']]

tfidf+faiss 43 fragmentos


,cita,seccion,puntaje
0,[SALUD],Naranja – Dañina para grupos sensibles (38 a 5...,0.291
1,[SALUD],Quiénes son los grupos sensibles,0.216
2,[SALUD],Rojo – Dañina a la salud (56 a 150 µg/m³),0.130


In [2]:
ev = json.loads((ROOT/'reports'/'eval_rag.json').read_text(encoding='utf-8'))
{k: {m: round(ev[k][m],2) for m in ('hit@1','hit@3','MRR')} for k in ('sin_expansion','con_expansion')}

{'sin_expansion': {'hit@1': 0.8, 'hit@3': 1.0, 'MRR': 0.89},
 'con_expansion': {'hit@1': 0.85, 'hit@3': 1.0, 'MRR': 0.93}}

## Herramientas (los errores vuelven como observación)

In [3]:
from aire.tools import call_tool
for name, args in [('get_pm25', {'estacion': 'Sabaneta', 'fecha': '2019-03-16', 'dias': 3}),
                   ('predict_tomorrow', {'estacion': 'ITA-CJUS', 'fecha': '2019-12-01'}),
                   ('get_pm25', {'estacion': 'Marte'})]:
    out = call_tool(name, args); print(name, '→', json.dumps(out, ensure_ascii=False)[:300], '\n')

get_pm25 → {"estacion": "Sabaneta - I.E. Rafael J. Mejía", "codigo": 90, "nombre_corto": "SAB-RAME", "municipio": "Sabaneta", "tipo_estacion": "Poblacional", "fecha_consultada": "2019-03-16", "pm25_promedio_24h": 52.34, "categoria_ica": "Dañina a la salud de grupos sensibles", "supera_norma_diaria_37": true, " 



predict_tomorrow → {"codigo": 28, "fecha_base": "2019-12-01", "fecha_pronostico": "2019-12-02", "pm25_fecha_base": 21.5, "estado_hmm_actual": "Moderado", "p_estado_hmm": {"Bajo": 0.003, "Moderado": 0.65, "Elevado": 0.348}, "clase_predicha": "Aceptable", "probabilidades": {"Buena": 0.019, "Aceptable": 0.9, "Dañina": 0. 

get_pm25 → {"error": "No encontré la estación 'Marte'. Estaciones válidas (código corto): GIR-SOSN, CEN-TRAF, MED-UNNV, ITA-CJUS, CAL-LASA, ITA-CONC, MED-LAYE, SUR-TRAF, CAL-JOAR, EST-HOSP, MED-ALTA, MED-VILL, BAR-TORR, COP-CVID, MED-BEME, MED-TESO, MED-SCRI, MED-ARAN, BEL-FEVE, ENV-HOSP, SAB-RAME. También pue 



## Agente

In [4]:
from aire import config as C
from aire.agent import build_agent
agent = build_agent(); print(type(agent).__name__, '| modelo:', C.LLM_MODEL if C.LLM_API_KEY else '(sin LLM)')
res = agent.run('¿Qué calidad del aire se espera mañana en Sabaneta y qué recomiendan?', context={'fecha': '2019-03-15'})
print(res.answer); print('\nHerramientas:', res.tools_used); print('Citas:', res.citations)

OfflineAgent | modelo: (sin LLM)


En SAB-RAME (Sabaneta) el PM2.5 promedio del 2019-03-15 fue 48.9 µg/m³, categoría ICA «Dañina a la salud de grupos sensibles» [SIATA-DATOS]. Para el 2019-03-16 el modelo pronostica «Dañina» (Buena 0%, Aceptable 25%, Dañina 75%); el estado latente actual es «Elevado» [MODELO]. Según «Naranja – Dañina para grupos sensibles (38 a 55 µg/m³)»: Adultos mayores, niños y personas con afecciones pulmonares o cardiovasculares pueden verse afectados si hacen actividades al aire libre por tiempo prolongado. Se recomienda a los grupos sensibles reducir o evitar el ejercicio vigoroso al aire libre, en especial en las horas de mayor tráfico (aproximadamente 6:00–10:00 y 16:00–20:00), y preferir el transporte público para no aumentar las emisiones. [SALUD] Esta información es orientativa y no es consejo médico.

Herramientas: ['get_pm25', 'predict_tomorrow', 'search_docs']
Citas: ['SIATA-DATOS', 'MODELO', 'SALUD']


## Evaluación con 15 preguntas fijas
Ejecutar `python evaluation/run_eval_agente.py --backend llm` (con clave) y `--backend offline`.

In [5]:
for b in ('offline', 'llm'):
    p = ROOT/'reports'/f'eval_agente_{b}.json'
    if p.exists(): print(b, json.loads(p.read_text(encoding='utf-8'))['resumen'])

offline {'backend': 'offline', 'n': 15, 'herramienta_correcta': 0.9333333333333333, 'cito_fuente': 0.8666666666666667, 'contenido_correcto': 0.8666666666666667, 'llamadas_herramientas_promedio': 1.7333333333333334, 'segundos_promedio': 0.07066666666666667, 'respaldos_sin_llm': None}
